# 15 · scikit-learn — notebook d'exercices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookIA/blob/main/chapitres/ch15_sklearn/03_notebook.ipynb)

La partie 0 vérifie tes réponses courtes aux quiz, aux rappels et aux exercices papier. Chaque exercice de code : un énoncé, une cellule à compléter (les `...` et les `raise NotImplementedError`), puis une cellule de vérification (`wb.check`). « Exécuter tout » va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳. Les questions « dans tes notes » qui n'ont pas de cellule 📝 se notent dans la section « Notes sur le notebook » de ta copie de `06_mes_reponses.md`. Bloqué 15 minutes ? `04_indices.md`.

> Travaille dans **ta copie** (`mon_travail/ch15_sklearn/03_notebook.ipynb`, créée par `python tools/start_chapter.py 15`) : ce fichier-ci est mis à jour par Claude.

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | 15.Q1 à 15.Q12, 15.R1 à 15.R3, 15.1 à 15.5 | vérifier tes réponses courtes | 🧠 🔁 ✏️ 🧮 | ★ à ★★ | |
| A | 15.9 à 15.12 | l'interface d'un estimateur, ses trois sorties, les datasets et le découpage, le code de 2018 à faire tourner avec la version 1.6 (*prochaine session de génération*) | | | |
| B | 15.13 à 15.16 | clustering (KMeans, DBSCAN, hiérarchique), transformateurs et inverse_transform, PCA et blanchiment, ensembles sur cinq spirales (*prochaine session de génération*) | | | |
| C | 15.17 à 15.24 | validation croisée, recherches en grille et au hasard, frontières de décision, pipeline polynomial, lecture de cv_results_, la fuite du ch. 8 refaite avec un pipeline (*prochaine session de génération*) | | | |
| D | 15.25 à 15.31 | seuil de décision, classes déséquilibrées, six familles de modèles, ColumnTransformer sur les manchots bruts, ta librairie face à scikit-learn, sauvegarde, le défi (*prochaine session de génération*) | | | |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (CPU, a few minutes); False = full version
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookIA.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookIA")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WB_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wb").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wb").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier du workbook.")

os.environ["WB_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wb  # noqa: E402

cfg = wb.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="15")  # your own library

## Objectifs et rappel express

- Utiliser l'interface commune de scikit-learn : hyperparamètres au constructeur, `fit` qui renvoie l'objet, attributs appris suffixés par `_`, `predict`, `decision_function`, `predict_proba`, `transform` et `inverse_transform`.
- Évaluer et régler un modèle : `cross_val_score`, `cross_validate`, `KFold`, `StratifiedKFold`, `GridSearchCV`, `RandomizedSearchCV`, et lire `cv_results_`.
- Enchaîner sans fuite : `Pipeline`, `make_pipeline`, `ColumnTransformer`, noms `étape__paramètre`.
- Moderniser du code ancien, choisir un seuil de décision, traiter des classes déséquilibrées, sauvegarder un pipeline avec ses versions.

**Rappel express.** $\hat p = \sigma(s) = 1/(1 + e^{-s})$ à deux classes, le softmax à $K$ classes ; nombre d'entraînements d'une grille : $\big(\prod_h n_h\big) \times k + 1$ ; probabilité qu'au moins un de $n_{\text{iter}}$ essais au hasard tombe dans le meilleur $q$ : $1 - (1 - q)^{n_{\text{iter}}}$ ; nombre de features polynomiales : $\binom{p + d}{d}$ ; min-max : $x' = (x - m)/(M - m)$. En scikit-learn : `sklearn.model_selection` (validation croisée, recherches, découpage), `sklearn.pipeline`, `sklearn.compose`, `sklearn.preprocessing`, `sklearn.inspection`, `sklearn.calibration` ; `joblib` pour la sauvegarde.

## Partie 0 · Vérifier tes réponses courtes (quiz, rappels, ✏️ 15.1 à 15.4 et 🧮 15.5)

Fais d'abord les quiz, les rappels et les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse courte ici : **la valeur** que tu as trouvée, pas l'expression Python, sinon tu ne vérifies rien. Un nombre : `42` ou `0.3` (en Python, le séparateur décimal est un **point** ; `0,3` sans guillemets serait un couple de deux nombres) ; un vrai ou faux : `True` ou `False` ; un choix : la lettre seule, entre guillemets (`"E"`) ; un nom : une chaîne (`"standardscaler"`) ; plusieurs nombres : une liste (`[4, 7]`). Arrondis comme l'énoncé le demande, et seulement à la fin du calcul. Les réponses pas encore remplies affichent ⏳. Les questions « dans ta copie », la réflexion (🗣️ 15.6, ⚖️ 15.7, 📄 15.8) et l'entretien se corrigent avec `05_solutions.md`.

**15.Q1 — Coder soi-même ou utiliser une bibliothèque ?**

In [ ]:
# 15.Q1: the values you found on paper
answer_15_Q1a = ...  # a) True or False
answer_15_Q1b = ...  # b) the letter of your choice, e.g. "E"
answer_15_Q1c = ...  # c) True or False
answer_15_Q1d = ...  # d) the letter of your choice, e.g. "E"

for letter, answer in zip("abcd", [answer_15_Q1a, answer_15_Q1b, answer_15_Q1c, answer_15_Q1d]):
    wb.check(f"15.Q1{letter}", answer)

**15.Q2 — Importer proprement et lire une dépréciation**

In [ ]:
# 15.Q2: the values you found on paper
answer_15_Q2a = ...  # a) the letter of your choice, e.g. "E"
answer_15_Q2b = ...  # b) the letter of your choice, e.g. "E"
answer_15_Q2c = ...  # c) True or False
answer_15_Q2d = ...  # d) the letter of your choice, e.g. "E"
answer_15_Q2e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_15_Q2a, answer_15_Q2b, answer_15_Q2c, answer_15_Q2d, answer_15_Q2e]):
    wb.check(f"15.Q2{letter}", answer)

**15.Q3 — Hyperparamètres au constructeur, attributs appris avec « _ »**

In [ ]:
# 15.Q3: the values you found on paper
answer_15_Q3a = ...  # a) the letter of your choice, e.g. "E"
answer_15_Q3b = ...  # b) True or False
answer_15_Q3c = ...  # c) True or False
answer_15_Q3d = ...  # d) the letter of your choice, e.g. "E"
answer_15_Q3e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_15_Q3a, answer_15_Q3b, answer_15_Q3c, answer_15_Q3d, answer_15_Q3e]):
    wb.check(f"15.Q3{letter}", answer)

**15.Q4 — Ce que renvoient fit et predict**

In [ ]:
# 15.Q4: the values you found on paper
answer_15_Q4a = ...  # a) the letter of your choice, e.g. "E"
answer_15_Q4b = ...  # b) the letter of your choice, e.g. "E"
answer_15_Q4c = ...  # c) the letter of your choice, e.g. "E"
answer_15_Q4d = ...  # d) True or False
answer_15_Q4e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_15_Q4a, answer_15_Q4b, answer_15_Q4c, answer_15_Q4d, answer_15_Q4e]):
    wb.check(f"15.Q4{letter}", answer)

**15.Q5 — decision_function ou predict_proba ?**

In [ ]:
# 15.Q5: the values you found on paper
answer_15_Q5a = ...  # a) the letter of your choice, e.g. "E"
answer_15_Q5b = ...  # b) the letter of your choice, e.g. "E"
answer_15_Q5c = ...  # c) the letter of your choice, e.g. "E"
answer_15_Q5d = ...  # d) a number
answer_15_Q5e = ...  # e) True or False
answer_15_Q5f = ...  # f) True or False

for letter, answer in zip("abcdef", [answer_15_Q5a, answer_15_Q5b, answer_15_Q5c, answer_15_Q5d, answer_15_Q5e, answer_15_Q5f]):
    wb.check(f"15.Q5{letter}", answer)

**15.Q6 — KMeans : n_clusters, labels_ et choix de k**

In [ ]:
# 15.Q6: the values you found on paper
answer_15_Q6a = ...  # a) a whole number
answer_15_Q6b = ...  # b) True or False
answer_15_Q6c = ...  # c) the letter of your choice, e.g. "E"
answer_15_Q6d = ...  # d) True or False
answer_15_Q6e = ...  # e) the letter of your choice, e.g. "E"

for letter, answer in zip("abcde", [answer_15_Q6a, answer_15_Q6b, answer_15_Q6c, answer_15_Q6d, answer_15_Q6e]):
    wb.check(f"15.Q6{letter}", answer)

**15.Q7 — fit, transform, fit_transform, inverse_transform**

In [ ]:
# 15.Q7: the values you found on paper
answer_15_Q7a = ...  # a) the letter of your choice, e.g. "E"
answer_15_Q7b = ...  # b) a number
answer_15_Q7c = ...  # c) a number
answer_15_Q7d = ...  # d) a number
answer_15_Q7e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_15_Q7a, answer_15_Q7b, answer_15_Q7c, answer_15_Q7d, answer_15_Q7e]):
    wb.check(f"15.Q7{letter}", answer)

**15.Q8 — Ce que la PCA fait vraiment aux features**

In [ ]:
# 15.Q8: the values you found on paper
answer_15_Q8a = ...  # a) the letter of your choice, e.g. "E"
answer_15_Q8b = ...  # b) True or False
answer_15_Q8c = ...  # c) True or False
answer_15_Q8d = ...  # d) True or False
answer_15_Q8e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_15_Q8a, answer_15_Q8b, answer_15_Q8c, answer_15_Q8d, answer_15_Q8e]):
    wb.check(f"15.Q8{letter}", answer)

**15.Q9 — Un ensemble est un estimateur comme les autres**

In [ ]:
# 15.Q9: the values you found on paper
answer_15_Q9a = ...  # a) True or False
answer_15_Q9b = ...  # b) the letter of your choice, e.g. "E"
answer_15_Q9c = ...  # c) the letter of your choice, e.g. "E"
answer_15_Q9d = ...  # d) True or False
answer_15_Q9e = ...  # e) the letter of your choice, e.g. "E"

for letter, answer in zip("abcde", [answer_15_Q9a, answer_15_Q9b, answer_15_Q9c, answer_15_Q9d, answer_15_Q9e]):
    wb.check(f"15.Q9{letter}", answer)

**15.Q10 — Grille ou recherche aléatoire ?**

In [ ]:
# 15.Q10: the values you found on paper
answer_15_Q10a = ...  # a) the letter of your choice, e.g. "E"
answer_15_Q10b = ...  # b) True or False
answer_15_Q10c = ...  # c) a whole number
answer_15_Q10d = ...  # d) a whole number
answer_15_Q10e = ...  # e) True or False
answer_15_Q10f = ...  # f) True or False

for letter, answer in zip("abcdef", [answer_15_Q10a, answer_15_Q10b, answer_15_Q10c, answer_15_Q10d, answer_15_Q10e, answer_15_Q10f]):
    wb.check(f"15.Q10{letter}", answer)

**15.Q11 — Pipeline : step__param et absence de fuite**

In [ ]:
# 15.Q11: the values you found on paper
answer_15_Q11a = ...  # a) the letter of your choice, e.g. "E"
answer_15_Q11b = ...  # b) True or False
answer_15_Q11c = ...  # c) the letter of your choice, e.g. "E"
answer_15_Q11d = ...  # d) the letter of your choice, e.g. "E"
answer_15_Q11e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_15_Q11a, answer_15_Q11b, answer_15_Q11c, answer_15_Q11d, answer_15_Q11e]):
    wb.check(f"15.Q11{letter}", answer)

**15.Q12 — Datasets, train_test_split et documentation**

In [ ]:
# 15.Q12: the values you found on paper
answer_15_Q12a = ...  # a) a number
answer_15_Q12b = ...  # b) True or False
answer_15_Q12c = ...  # c) the letter of your choice, e.g. "E"
answer_15_Q12d = ...  # d) True or False
answer_15_Q12e = ...  # e) a whole number
answer_15_Q12f = ...  # f) the letter of your choice, e.g. "E"

for letter, answer in zip("abcdef", [answer_15_Q12a, answer_15_Q12b, answer_15_Q12c, answer_15_Q12d, answer_15_Q12e, answer_15_Q12f]):
    wb.check(f"15.Q12{letter}", answer)

**15.R1 — Forêt aléatoire ou gradient boosting ?**

In [ ]:
# 15.R1: the values you found on paper
answer_15_R1a = ...  # a) the letter of your choice, e.g. "E"
answer_15_R1b = ...  # b) True or False
answer_15_R1c = ...  # c) the letter of your choice, e.g. "E"
answer_15_R1d = ...  # d) the letter of your choice, e.g. "E"
answer_15_R1e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_15_R1a, answer_15_R1b, answer_15_R1c, answer_15_R1d, answer_15_R1e]):
    wb.check(f"15.R1{letter}", answer)

**15.R2 — Où ajuster le scaler dans une validation croisée ?**

In [ ]:
# 15.R2: the values you found on paper
answer_15_R2a = ...  # a) True or False
answer_15_R2b = ...  # b) the letter of your choice, e.g. "E"
answer_15_R2c = ...  # c) a number
answer_15_R2d = ...  # d) True or False
answer_15_R2e = ...  # e) the letter of your choice, e.g. "E"

for letter, answer in zip("abcde", [answer_15_R2a, answer_15_R2b, answer_15_R2c, answer_15_R2d, answer_15_R2e]):
    wb.check(f"15.R2{letter}", answer)

**15.R3 — Combien de modèles pour une validation croisée à 5 folds ?**

In [ ]:
# 15.R3: the values you found on paper
answer_15_R3a = ...  # a) a whole number
answer_15_R3b = ...  # b) a whole number
answer_15_R3c = ...  # c) a whole number
answer_15_R3d = ...  # d) a whole number
answer_15_R3e = ...  # e) a whole number
answer_15_R3f = ...  # f) True or False

for letter, answer in zip("abcdef", [answer_15_R3a, answer_15_R3b, answer_15_R3c, answer_15_R3d, answer_15_R3e, answer_15_R3f]):
    wb.check(f"15.R3{letter}", answer)

**Ex 15.1 — Compter les modèles entraînés par une recherche en grille**

In [ ]:
# Ex 15.1: the values you found on paper
answer_15_1a = ...  # a) a whole number
answer_15_1b = ...  # b) a whole number
answer_15_1c = ...  # c) a whole number
answer_15_1d = ...  # d) a whole number
answer_15_1e = ...  # e) a whole number
answer_15_1f = ...  # f) a whole number
answer_15_1g = ...  # g) a whole number
answer_15_1h = ...  # h) a whole number

for letter, answer in zip("abcdefgh", [answer_15_1a, answer_15_1b, answer_15_1c, answer_15_1d, answer_15_1e, answer_15_1f, answer_15_1g, answer_15_1h]):
    wb.check(f"15.1{letter}", answer)

**Ex 15.2 — De decision_function à predict_proba**

In [ ]:
# Ex 15.2: the values you found on paper
answer_15_2a = ...  # a) a number with 4 decimals
answer_15_2b = ...  # b) a number with 4 decimals
answer_15_2c = ...  # c) a number
answer_15_2d = ...  # d) a list of 5 classes, e.g. [1, 1, 1, 1, 1]
answer_15_2e = ...  # e) a number with 4 decimals
answer_15_2f = ...  # f) a number with 4 decimals
answer_15_2g = ...  # g) a whole number
answer_15_2h = ...  # h) a number with 4 decimals
answer_15_2i = ...  # i) True or False
answer_15_2j = ...  # j) True or False

for letter, answer in zip("abcdefghij", [answer_15_2a, answer_15_2b, answer_15_2c, answer_15_2d, answer_15_2e, answer_15_2f, answer_15_2g, answer_15_2h, answer_15_2i, answer_15_2j]):
    wb.check(f"15.2{letter}", answer)

**Ex 15.3 — Ramener une droite ajustée dans l'espace d'origine**

In [ ]:
# Ex 15.3: the values you found on paper
answer_15_3a = ...  # a) a number
answer_15_3b = ...  # b) a number
answer_15_3c = ...  # c) a number
answer_15_3d = ...  # d) a number
answer_15_3e = ...  # e) a number
answer_15_3f = ...  # f) a number
answer_15_3g = ...  # g) a number
answer_15_3h = ...  # h) a number
answer_15_3i = ...  # i) a number
answer_15_3j = ...  # j) True or False

for letter, answer in zip("abcdefghij", [answer_15_3a, answer_15_3b, answer_15_3c, answer_15_3d, answer_15_3e, answer_15_3f, answer_15_3g, answer_15_3h, answer_15_3i, answer_15_3j]):
    wb.check(f"15.3{letter}", answer)

**Ex 15.4 — Combien de features polynomiales ?**

In [ ]:
# Ex 15.4: the values you found on paper
answer_15_4a = ...  # a) a whole number
answer_15_4b = ...  # b) a whole number
answer_15_4c = ...  # c) a whole number
answer_15_4d = ...  # d) a whole number
answer_15_4e = ...  # e) a whole number
answer_15_4f = ...  # f) a whole number
answer_15_4g = ...  # g) a whole number
answer_15_4h = ...  # h) a number with 2 decimals
answer_15_4i = ...  # i) a whole number

for letter, answer in zip("abcdefghi", [answer_15_4a, answer_15_4b, answer_15_4c, answer_15_4d, answer_15_4e, answer_15_4f, answer_15_4g, answer_15_4h, answer_15_4i]):
    wb.check(f"15.4{letter}", answer)

**Ex 15.5 — Budget d'une recherche d'hyperparamètres**

In [ ]:
# Ex 15.5: the values you found on paper
answer_15_5a = ...  # a) a whole number
answer_15_5b = ...  # b) a number with 1 decimal
answer_15_5c = ...  # c) a number with 1 decimal
answer_15_5d = ...  # d) a number with 1 decimal
answer_15_5e = ...  # e) a number with 4 decimals
answer_15_5f = ...  # f) a whole number
answer_15_5g = ...  # g) a number
answer_15_5h = ...  # h) a number with 1 decimal

for letter, answer in zip("abcdefgh", [answer_15_5a, answer_15_5b, answer_15_5c, answer_15_5d, answer_15_5e, answer_15_5f, answer_15_5g, answer_15_5h]):
    wb.check(f"15.5{letter}", answer)

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu dire ce que renvoient `fit`, `predict`, `decision_function`, `predict_proba` et `transform`, et sous quelle forme ?
2. Sais-tu construire un pipeline qui se règle par une recherche en grille ou au hasard sans fuite, et estimer le coût de cette recherche ?
3. Sais-tu moderniser un appel déprécié en lisant la documentation de ta version, et sauvegarder un modèle avec ses versions ?

**Pour aller plus loin** : les articles de Pedregosa et coll. (2011), de Buitinck et coll. (2013) et de Bergstra et Bengio (2012), la page « Common pitfalls » de la documentation, cités dans la fiche. La suite : le checkpoint de la partie III (ML classique), puis les réseaux de neurones (ch. 16).

*Les exercices de code 15.9 à 15.31 (parties A à D) seront ajoutés à la prochaine session de génération ; relance alors `python tools/start_chapter.py 15` pour obtenir le notebook complet.*